# ¿Es el polvo el que imita la escala λ? Prueba de sistemáticos

**Proyecto:** *Vacuum Particle Journey*, comprobación de sistemáticos.

**Contexto.** La validación independiente no confirmó la escala λ: con λ congelada no predice mejor las galaxias reservadas (p = 0.34). Pero la mejora que aporta λ se correlaciona con la **inclinación** (ρ = 0.25, p = 0.002), con el **muestreo** de la curva y con una **baja fracción de gas**. Ese patrón encaja con un efecto del **polvo**:
- en galaxias muy inclinadas, el polvo oculta la luz óptica del disco interior y puede deformar las curvas medidas con Hα;
- la emisión de polvo templado puede contaminar la luz infrarroja de 3.6 μm con la que SPARC estima la masa estelar;
- las galaxias pobres en gas y de tipos más tempranos suelen tener más polvo.

**Por qué esta prueba protege la idea:** si el polvo explica la señal de λ, se descarta una pista falsa, no la hipótesis de fondo (una corrección geométrica no local del entrelazamiento). Si no la explica, la pista de las curvas bien muestreadas gana peso.

**Hipótesis fijadas antes de calcular los indicadores de polvo** (la inclinación ya se vio en la validación; el resto no):

| | Si la señal es un artefacto del polvo | Si la señal es física |
|---|---|---|
| P1 | La mejora crece con la inclinación, con tipos tempranos (más polvo), con la luminosidad y con datos Hα en el interior | No depende de esos indicadores |
| P2 | En una **submuestra limpia** (inclinación < 75° y sin datos ópticos interiores), λ se reduce y deja de predecir el grupo reservado | En la submuestra limpia, λ se mantiene y predice el grupo reservado |
| P3 | La pista de "curvas bien muestreadas" desaparece al controlar por polvo | La pista se mantiene al controlar por polvo |

**Indicador de datos ópticos (Hα) en el interior:** las curvas de radio (HI) tienen un haz de ~10–30″, así que no muestrean más fino que unos pocos segundos de arco. Si los primeros puntos de una curva están separados menos de **4″**, casi seguro vienen de datos ópticos, que el polvo sí afecta. Es un indicador aproximado, sin acceso a la fuente exacta de cada punto.

In [ ]:
import os, io, re, glob, zipfile, urllib.request, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares, minimize_scalar
from scipy.special import i0, i1, k0, k1
from scipy.stats import spearmanr

warnings.filterwarnings("ignore", category=RuntimeWarning)
RNG = np.random.default_rng(42)
QUICK = False

COL = {"Datos": "#222222", "Solo bariones": "#8a8a85", "NFW ΛCDM (c–M)": "#6250d6",
       "RAR (g† = 1.2e-10)": "#e87ba4", "RAR local (g† global)": "#eb6834", "RAR no local (ER)": "#1baf7a", "Instrumental": "#2a78d6", "Densidad": "#eda100", "Tamaño": "#e34948"}
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.6,
    "axes.edgecolor": "#c3c2b7", "lines.linewidth": 2, "font.size": 10, "legend.frameon": False,
})

G = 4.30091e-6
C_KMS = 299792.458
H0 = 70.0; H0_KPC = H0 / 1000.0
KPC_M = 3.0857e19
TO_MS2 = 1e6 / KPC_M                    # (km/s)^2/kpc -> m/s^2
try:
    import plotly.graph_objects as go
    HAVE_PLOTLY = True
except ImportError:
    HAVE_PLOTLY = False
    print("plotly no disponible: las figuras 3D se harán con matplotlib (estáticas)")
from scipy import fft as sfft

import hashlib, platform, json, scipy
from scipy.stats import wilcoxon
SEED = 20261119
RNG = np.random.default_rng(SEED)
SPLIT_SALT = "vpj-validacion-2026"
ALPHA_Q = 0.27            # λ de MOND completo ≈ 0.27·R_d (resultado del notebook 6)

## 1. Datos, modelo y división (idénticos a la validación independiente)

In [ ]:
DATA_DIR = "sparc_rotmod"
BASES = ["https://astroweb.case.edu/SPARC/", "http://astroweb.cwru.edu/SPARC/"]
ZENODO = "https://zenodo.org/records/16284118/files/Rotmod_LTG.zip?download=1"

def _get(url):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    return urllib.request.urlopen(req, timeout=60).read()

def _files():
    return sorted(glob.glob(os.path.join(DATA_DIR, "**", "*_rotmod.dat"), recursive=True))

def get_sparc():
    os.makedirs(DATA_DIR, exist_ok=True)
    if _files():
        return True
    for url in [b + "Rotmod_LTG.zip" for b in BASES] + [ZENODO]:
        try:
            print("Descargando", url)
            zipfile.ZipFile(io.BytesIO(_get(url))).extractall(DATA_DIR)
            if _files():
                return True
        except Exception as e:
            print("  falló:", type(e).__name__)
    try:
        from google.colab import files
        print("Sube Rotmod_LTG.zip (de astroweb.case.edu/SPARC)")
        for name, content in files.upload().items():
            zipfile.ZipFile(io.BytesIO(content)).extractall(DATA_DIR)
        return bool(_files())
    except Exception:
        return False

def get_quality():
    for b in BASES:
        try:
            txt = _get(b + "SPARC_Lelli2016c.mrt").decode("latin-1")
            q = {}
            for line in txt.splitlines():
                t = line.split()
                if len(t) >= 18 and re.fullmatch(r"\d+", t[1] or "") and t[17] in ("1", "2", "3"):
                    q[t[0]] = dict(Q=int(t[17]), Inc=float(t[5]))
            if len(q) > 100:
                return q
        except Exception:
            pass
    return None

def load_rotmod(path):
    name = os.path.basename(path).replace("_rotmod.dat", "")
    d = np.atleast_2d(np.loadtxt(path, comments="#"))
    g = dict(name=name, r=d[:, 0], vobs=d[:, 1], err=d[:, 2], vgas=d[:, 3], vdisk=d[:, 4], vbul=d[:, 5])
    ok = (g["r"] > 0) & (g["err"] > 0)
    o = np.argsort(g["r"][ok])
    for k in ["r", "vobs", "err", "vgas", "vdisk", "vbul"]:
        g[k] = g[k][ok][o]
    g["vgas2"] = g["vgas"] * np.abs(g["vgas"])
    g["vstar2"] = g["vdisk"] * np.abs(g["vdisk"]) + 1.4 * g["vbul"] * np.abs(g["vbul"])
    return g

def _v2_expdisk(r, M, Rd):
    y = r / (2 * Rd)
    return 4 * np.pi * G * (M / (2 * np.pi * Rd**2)) * Rd * y**2 * (i0(y) * k0(y) - i1(y) * k1(y))

def _v2_nfw(r, V200, c):
    x = r / (V200 / (10 * H0_KPC))
    mu = lambda y: np.log1p(y) - y / (1 + y)
    return V200**2 * mu(c * x) / (x * mu(c))

def make_synthetic_set(n=30, seed=0):
    rng = np.random.default_rng(seed)
    out = {}
    for name in ["NGC3198", "NGC2403", "DDO154"] + [f"SYN{i:03d}" for i in range(n - 3)]:
        V200 = 10 ** rng.uniform(1.6, 2.4); c = 10 ** rng.normal(1.0, 0.12)
        Rd = rng.uniform(1, 5); Ml = 10 ** rng.uniform(8.5, 11); Mg = Ml * 10 ** rng.uniform(-1, 0.7)
        r = np.linspace(0.3 * Rd, rng.uniform(5, 9) * Rd, rng.integers(12, 40))
        vdisk = np.sqrt(_v2_expdisk(r, Ml, Rd)); vgas = np.sqrt(_v2_expdisk(r, Mg, 2.5 * Rd))
        vtrue = np.sqrt(vgas**2 + 0.5 * vdisk**2 + _v2_nfw(r, V200, c))
        err = 2 + 0.04 * vtrue
        os.makedirs("synthetic", exist_ok=True)
        path = os.path.join("synthetic", f"{name}_rotmod.dat")
        np.savetxt(path, np.c_[r, vtrue + rng.normal(0, err), err, vgas, vdisk, 0 * r], header="Distance = 10 Mpc")
        out[name] = load_rotmod(path)
    return out

HAVE_SPARC = get_sparc()
if HAVE_SPARC:
    GAL = {}
    for f in _files():
        g = load_rotmod(f)
        if len(g["r"]) >= 5:
            GAL[g["name"]] = g
    Q = get_quality()
    if Q:
        GAL = {k: v for k, v in GAL.items() if k in Q and Q[k]["Q"] < 3 and Q[k]["Inc"] >= 30}
        print("Corte de calidad aplicado (Q<3, inc≥30°).")
    else:
        print("Sin tabla de calidad: se usan todas las galaxias.")
else:
    print("⚠️  SIN DATOS REALES: galaxias SINTÉTICAS con halo NFW. Los resultados no tienen valor físico.")
    GAL = make_synthetic_set()

if QUICK and len(GAL) > 40:
    keep = [k for k in ["NGC3198", "NGC2403", "DDO154"] if k in GAL]
    rest = [k for k in GAL if k not in keep]
    GAL = {k: GAL[k] for k in keep + list(RNG.choice(rest, 40 - len(keep), replace=False))}
N_TOT = sum(len(g["r"]) for g in GAL.values())
print(f"Galaxias: {len(GAL)} | puntos: {N_TOT}")

In [ ]:
NG = 400
U_GRID = np.geomspace(0.1, 1.5, 40)
LAM_GRID = np.r_[0.0, np.geomspace(0.1, 100, 24)]                                       # kpc
GD_GRID = np.unique(np.r_[np.geomspace(0.6e-10, 2.4e-10, 25), 1.2e-10]) / TO_MS2        # (km/s)^2/kpc
I_GD12 = int(np.argmin(np.abs(GD_GRID * TO_MS2 - 1.2e-10)))

def vb2(g, u):
    return g["vgas2"] + u * g["vstar2"]

def prior(u):
    return np.log10(u / 0.5) / 0.1

def rar_g(gb, gd):
    return gb / (-np.expm1(-np.sqrt(gb / gd)))

def F_matrix(r, a, lam):
    R, A = r[:, None], a[None, :]
    if lam == 0:
        return (A < R).astype(float)
    x = R / lam
    e_lo = np.exp(-np.clip((A - R) / lam, 0, None))
    e_hi = np.exp(-np.clip((R - A) / lam, 0, None))
    e_sum = np.exp(-(A + R) / lam)
    I_lo = e_lo * (x - 1) + e_sum * (1 + x)
    I_hi = 2 * A / lam - e_hi * (1 + x) + e_sum * (1 + x)
    return np.where(R <= A, I_lo, I_hi) * lam / (2 * A)

def setup(g):
    if "_s" in g:
        return g["_s"]
    rmax = g["r"].max()
    rg = np.linspace(rmax / NG, rmax, NG)
    s = dict(rg=rg, mids=0.5 * (np.r_[0.0, rg[:-1]] + rg),
             vg=np.interp(rg, np.r_[0.0, g["r"]], np.r_[0.0, g["vgas2"]]),
             vs=np.interp(rg, np.r_[0.0, g["r"]], np.r_[0.0, g["vstar2"]]))
    g["_s"] = s
    return s

def phantom_shells(s, U, GD):
    # G·M_fantasma en cascarones -> forma (nΥ, ng†, NG)
    v2 = s["vg"][None, :] + U[:, None] * s["vs"][None, :]
    gb = np.clip(v2, 1e-6, None) / s["rg"][None, :]
    gr = rar_g(gb[:, None, :], GD[None, :, None])
    GM = s["rg"][None, None, :] ** 2 * (gr - gb[:, None, :])
    return np.diff(np.concatenate([np.zeros(GM.shape[:2] + (1,)), GM], axis=2), axis=2)

def model_v2(g, u, lam, gd):
    s = setup(g)
    d = phantom_shells(s, np.array([u]), np.array([gd]))[0, 0]
    return vb2(g, u) + F_matrix(g["r"], s["mids"], lam) @ d / g["r"]

# Comprobaciones: F(λ→0) es un escalón; con λ pequeño se recupera la RAR local exacta
_g = next(iter(GAL.values()))
v_loc = np.sqrt(_g["r"] * rar_g(np.clip(vb2(_g, 0.5), 1e-6, None) / _g["r"], 1.2e-10 / TO_MS2))
v_0 = np.sqrt(model_v2(_g, 0.5, 0.0, 1.2e-10 / TO_MS2))
v_s = np.sqrt(model_v2(_g, 0.5, 1e-3, 1.2e-10 / TO_MS2))
print(f"Test: |v(λ=0) − RAR exacta| máx = {np.max(np.abs(v_0 - v_loc)):.2f} km/s; "
      f"|v(λ=0.001) − v(λ=0)| máx = {np.max(np.abs(v_s - v_0)):.3f} km/s (ambos deben ser ≈ 0)")

In [ ]:
ARCSEC = 206265.0
paths = _files() if HAVE_SPARC else glob.glob(os.path.join("synthetic", "*_rotmod.dat"))
DIST = {}
for f in paths:
    with open(f) as fh:
        for line in fh:
            m = re.search(r"Distance\s*=\s*([\d.]+)", line)
            if m:
                DIST[os.path.basename(f).replace("_rotmod.dat", "")] = float(m.group(1)); break
if not HAVE_SPARC:
    _rd = np.random.default_rng(3)
    DIST = {k: float(10 ** _rd.uniform(0.5, 1.8)) for k in GAL}
    print("⚠️  Distancias sintéticas aleatorias (solo para probar el código)")
GAL = {k: g for k, g in GAL.items() if k in DIST}
NAMES = list(GAL)
D = np.array([DIST[k] for k in NAMES])
RMAX = np.array([GAL[k]["r"].max() for k in NAMES])
MB = np.array([max(GAL[k]["r"][-1] * vb2(GAL[k], 0.5)[-1] / G, 1e5) for k in NAMES])
SIGMA = MB / (np.pi * (RMAX * 1e3) ** 2)          # M☉/pc²
N_TOT_PTS = sum(len(g["r"]) for g in GAL.values())
print(f"{len(NAMES)} galaxias | D: {D.min():.1f}–{D.max():.1f} Mpc | Σ_b: {np.median(SIGMA):.1f} M☉/pc² (mediana)")
rho, p = spearmanr(np.log10(D), np.log10(RMAX))
print(f"Correlación distancia–tamaño en la muestra: ρ = {rho:.2f} (p = {p:.0e})  ← por esto λ instrumental imitaría 'λ ∝ tamaño'")

In [ ]:
def get_rdisk():
    for b in BASES:
        try:
            txt = _get(b + "SPARC_Lelli2016c.mrt").decode("latin-1"); out = {}
            for line in txt.splitlines():
                t = line.split()
                if len(t) >= 18 and re.fullmatch(r"\d+", t[1] or "") and t[17] in ("1", "2", "3"):
                    out[t[0]] = float(t[11])
            if len(out) > 100:
                return out
        except Exception:
            pass
    return {}

RD_TAB = get_rdisk()
RD = []
for k in NAMES:
    g = GAL[k]
    est = g["r"][np.argmax(np.abs(g["vdisk"]))] / 2.2 if np.any(g["vdisk"] != 0) else np.nan
    RD.append(RD_TAB.get(k, est) if RD_TAB.get(k, 0) > 0 else est)
RD = np.array(RD)
RD = np.where(np.isfinite(RD) & (RD > 0), RD, np.nanmedian(RD))
print(f"R_d {'de la tabla de SPARC' if RD_TAB else 'estimado desde V_disk'}: mediana {np.median(RD):.2f} kpc "
      f"(16–84 %: {np.percentile(RD,16):.2f}–{np.percentile(RD,84):.2f})")

In [ ]:
def group_of(name):
    return "descubrimiento" if int(hashlib.md5((SPLIT_SALT + name).encode()).hexdigest(), 16) % 2 == 0 else "reservada"
GROUP = np.array([group_of(k) for k in NAMES])
DISC, HOLD = np.where(GROUP == "descubrimiento")[0], np.where(GROUP == "reservada")[0]
print(f"Descubrimiento: {len(DISC)} galaxias | Reservada: {len(HOLD)} galaxias")
print(f"Descubrimiento: {len(DISC)} | Reservada: {len(HOLD)}  (misma división que en la validación)")

## 2. Indicadores de polvo para cada galaxia

De la tabla general de SPARC: **inclinación**, **tipo de Hubble T** (0 = S0, 3 = Sb, 5 = Sc, 7–10 = Sd/irregulares; los tipos tempranos tienen más polvo y metales), **luminosidad a 3.6 μm** (las galaxias más luminosas son más metálicas y polvorientas) y **masa de HI**. De cada curva: el **espaciado angular de los puntos interiores**, para el indicador Hα.

In [ ]:
def get_table1():
    for b in BASES:
        try:
            txt = _get(b + "SPARC_Lelli2016c.mrt").decode("latin-1"); out = {}
            for line in txt.splitlines():
                t = line.split()
                if len(t) >= 18 and re.fullmatch(r"\d+", t[1] or "") and t[17] in ("1", "2", "3"):
                    out[t[0]] = dict(T=int(t[1]), Inc=float(t[5]), L36=float(t[7]), MHI=float(t[13]), Q=int(t[17]), Ref=" ".join(t[18:]))
            if len(out) > 100:
                return out
        except Exception:
            pass
    return None

T1 = get_table1()
if T1 is None:
    print("⚠️  Sin tabla de SPARC: indicadores SINTÉTICOS aleatorios (solo para probar el código)")
    _r = np.random.default_rng(5)
    T1 = {k: dict(T=int(_r.integers(0, 11)), Inc=float(_r.uniform(30, 90)), L36=float(10 ** _r.uniform(-1, 2)),
                  MHI=float(10 ** _r.uniform(-1, 1)), Q=int(_r.integers(1, 3)), Ref="sintético") for k in NAMES}
INC = np.array([T1[k]["Inc"] if k in T1 else np.nan for k in NAMES])
TYPE = np.array([T1[k]["T"] if k in T1 else np.nan for k in NAMES])
LOGL = np.log10(np.array([max(T1[k]["L36"], 1e-3) if k in T1 else np.nan for k in NAMES]))
GASF = np.array([T1[k]["MHI"] * 1.33 / (T1[k]["MHI"] * 1.33 + 0.5 * T1[k]["L36"]) if k in T1 else np.nan for k in NAMES])
def inner_spacing_arcsec(g, Dm):
    th = g["r"] / (Dm * 1e3) * ARCSEC
    return float(np.median(np.diff(th[: min(6, len(th))]))) if len(th) > 2 else np.nan
SPAC = np.array([inner_spacing_arcsec(GAL[k], D[i]) for i, k in enumerate(NAMES)])
NPTS = np.array([len(GAL[k]["r"]) for k in NAMES])
HALPHA = SPAC < 4.0
DUSTY_INC = INC >= 75
CLEAN = (~DUSTY_INC) & (~HALPHA)
print(f"Con datos ópticos en el interior (espaciado < 4″): {HALPHA.sum()} de {len(NAMES)}")
print(f"Muy inclinadas (≥ 75°): {DUSTY_INC.sum()} | Submuestra limpia: {CLEAN.sum()} "
      f"(descubrimiento {CLEAN[DISC].sum()}, reservada {CLEAN[HOLD].sum()})")

## 3. Tabla de χ² y mejora de cada galaxia

Misma tabla que en la validación (geometría esférica, 5 % de error sistemático). La mejora de cada galaxia, $\Delta\chi^2_i/N_i$, se mide con λ y g† ajustados en el grupo de descubrimiento completo, igual que en la validación.

In [ ]:
LAM_GRID = np.r_[0.0, np.geomspace(0.02, 200, 45)]
LOGLAM = np.log(np.r_[0.01, LAM_GRID[1:]])
BEAM_ARCSEC = 15.0
def chi_table(floor=0.0, cut_beams=None):
    TOT = np.zeros((len(NAMES), len(LAM_GRID), len(GD_GRID))); DAT = np.zeros_like(TOT); NPT = np.zeros(len(NAMES), int)
    for n_i, name in enumerate(NAMES):
        g = GAL[name]; s = setup(g)
        err = np.sqrt(g["err"] ** 2 + (floor * g["vobs"]) ** 2)
        mask = np.ones(len(g["r"]), bool)
        if cut_beams:
            mask = g["r"] >= cut_beams * BEAM_ARCSEC / ARCSEC * D[n_i] * 1e3
            if mask.sum() < 5:
                mask = np.zeros(len(g["r"]), bool); mask[-5:] = True
        NPT[n_i] = mask.sum()
        d = phantom_shells(s, U_GRID, GD_GRID)
        base = vb2(g, U_GRID[:, None]); pr = prior(U_GRID) ** 2
        for l_i, lam in enumerate(LAM_GRID):
            extra = np.einsum("in,ugn->ugi", F_matrix(g["r"], s["mids"], lam), d) / g["r"]
            v = np.sqrt(np.clip(base[:, None, :] + extra, 0, None))
            c = ((((g["vobs"] - v) / err) ** 2) * mask).sum(-1)
            t = c + pr[:, None]; k = t.argmin(0); a = np.arange(len(GD_GRID))
            TOT[n_i, l_i] = t[k, a]; DAT[n_i, l_i] = c[k, a]
    return dict(tot=TOT, dat=DAT, npts=NPT)



t0 = time.time()
TAB = chi_table(floor=0.05)
print(f"Tabla: {time.time()-t0:.0f} s")

def per_gal(lam_vec, which="tot"):
    T = TAB[which]; n = np.arange(T.shape[0])
    ll = np.log(np.clip(lam_vec, 0.01, 200))
    k = np.clip(np.searchsorted(LOGLAM, ll), 1, len(LOGLAM) - 1)
    w = np.clip((ll - LOGLAM[k - 1]) / (LOGLAM[k] - LOGLAM[k - 1]), 0, 1)[:, None]
    return T[n, k - 1, :] * (1 - w) + T[n, k, :] * w

def fit_on(idx):
    S = TAB["tot"][idx].sum(0)
    i, j = np.unravel_index(np.argmin(S), S.shape)
    j0 = int(np.argmin(S[0]))
    Sm = per_gal(ALPHA_Q * RD)[idx].sum(0); jm = int(np.argmin(Sm))
    return dict(i=i, j=j, j0=j0, jm=jm, lam=float(LAM_GRID[i]))

def gains(f):
    d_loc = TAB["dat"][:, 0, f["j0"]] - TAB["dat"][:, f["i"], f["j"]]
    d_mond = per_gal(ALPHA_Q * RD, "dat")[:, f["jm"]] - TAB["dat"][:, f["i"], f["j"]]
    return d_loc, d_mond

F_ALL = fit_on(DISC)
D_LOC, D_MOND = gains(F_ALL)
DPP = D_LOC / NPTS
print(f"λ con todo el grupo de descubrimiento: {F_ALL['lam']:.3g} kpc (validación: ~1.07)")

## 4. P1: ¿depende la mejora de los indicadores de polvo?

Correlaciones una a una y una **regresión múltiple** con todos los indicadores a la vez (variables estandarizadas). La regresión separa efectos que van juntos: por ejemplo, las galaxias luminosas también suelen ser de tipos tempranos.

In [ ]:
from scipy.stats import mannwhitneyu
PROX = {"inclinación": INC, "tipo de Hubble T (bajo = más polvo)": TYPE, "log luminosidad 3.6 μm": LOGL,
        "espaciado interior [″] (bajo = Hα)": np.log10(np.clip(SPAC, 0.1, None)), "fracción de gas": GASF, "log número de puntos": np.log10(NPTS)}
rows = []
for nm, v in PROX.items():
    ok = np.isfinite(v) & np.isfinite(DPP)
    rho, p = spearmanr(v[ok], DPP[ok]); rows.append(dict(indicador=nm, ρ=round(rho, 2), p=f"{p:.1e}"))
P1 = pd.DataFrame(rows)
print(P1.to_string(index=False))

ok = np.all([np.isfinite(v) for v in PROX.values()], axis=0) & np.isfinite(DPP)
Xr = np.c_[[PROX[k][ok] for k in PROX]].T; Xs = (Xr - Xr.mean(0)) / Xr.std(0)
y = DPP[ok]; A = np.c_[np.ones(ok.sum()), Xs]
coef, *_ = np.linalg.lstsq(A, y, rcond=None); res = y - A @ coef
se = np.sqrt(np.diag(np.linalg.inv(A.T @ A)) * (res @ res) / max(ok.sum() - A.shape[1], 1))
REG = pd.DataFrame(dict(coef=coef[1:], err=se[1:], t=coef[1:] / se[1:]), index=list(PROX)).round(3)
print("\nRegresión múltiple de la mejora por punto (|t| > 3 = efecto claro):"); print(REG.to_string())

u = mannwhitneyu(DPP[HALPHA | DUSTY_INC], DPP[CLEAN], alternative="greater")
print(f"\nMejora media por punto: galaxias con riesgo de polvo {np.mean(DPP[HALPHA | DUSTY_INC]):.3f} | "
      f"limpias {np.mean(DPP[CLEAN]):.3f} | Mann–Whitney (riesgo > limpias) p = {u.pvalue:.1e}")

fig, axs = plt.subplots(1, 3, figsize=(13, 3.8), sharey=True)
for ax, (nm, v) in zip(axs, [("inclinación [°]", INC), ("tipo de Hubble T", TYPE), ("espaciado interior [″]", SPAC)]):
    ax.plot(v[~CLEAN], DPP[~CLEAN], "o", ms=4, color="#e34948", alpha=0.7, label="riesgo de polvo")
    ax.plot(v[CLEAN], DPP[CLEAN], "o", ms=4, color="#1baf7a", alpha=0.7, label="limpias")
    ax.axhline(0, color="#c3c2b7", lw=1); ax.set_xlabel(nm)
    if "espaciado" in nm: ax.set_xscale("log"); ax.axvline(4, color="#9a9a95", ls="--", lw=1)
axs[0].set_ylabel("mejora por punto con λ (Δχ²/N)"); axs[0].legend(fontsize=8.5)
axs[0].set_title("¿Dónde mejora λ el ajuste?", loc="left")
plt.tight_layout(); plt.show()

## 5. P2: la prueba decisiva en la submuestra limpia

Se repite la validación **solo con galaxias limpias**: λ y g† se ajustan con las limpias del grupo de descubrimiento, y se predicen las limpias del grupo reservado con los parámetros congelados. Se comparan con la RAR local y con MOND completo (λ = 0.27·R_d).

In [ ]:
CD, CH = DISC[CLEAN[DISC]], HOLD[CLEAN[HOLD]]
F_CL = fit_on(CD)
dl, dm = gains(F_CL)
rows = []
for nm, d in [("RAR local (λ = 0)", dl[CH]), ("MOND completo (0.27·R_d)", dm[CH])]:
    st = wilcoxon(d, alternative="greater") if np.any(d != 0) else None
    rows.append(dict(rival=nm, Δχ2_total=round(d.sum(), 1), mediana=round(float(np.median(d)), 2),
                     galaxias_mejor=f"{np.mean(d > 0):.0%}", p_Wilcoxon=f"{st.pvalue:.1e}" if st else "–"))
P2 = pd.DataFrame(rows)
bl = np.array([LAM_GRID[np.unravel_index(np.argmin(TAB["tot"][RNG.choice(CD, len(CD))].sum(0)), TAB["tot"].shape[1:])[0]] for _ in range(1000)])
print(f"Submuestra limpia: λ = {F_CL['lam']:.3g} kpc (bootstrap 68 %: {np.percentile(bl,16):.2f}–{np.percentile(bl,84):.2f}; "
      f"λ = 0 en {np.mean(bl==0):.0%}) frente a {F_ALL['lam']:.3g} kpc con todas")
print(f"Grupo reservado limpio ({len(CH)} galaxias), positivo = gana λ:")
print(P2.to_string(index=False))

## 6. P3: ¿sobrevive la pista de las curvas bien muestreadas?

En la validación, λ mejoraba en ambos grupos las galaxias con ≥ 30 puntos (p ≈ 0.02–0.04, análisis hecho a posteriori). Aquí se mira si esa pista se mantiene **dentro de la submuestra limpia**, y si el número de puntos sigue importando en la regresión cuando se controla por los indicadores de polvo.

In [ ]:
rows = []
for nm, idx in [("todas, ≥ 30 puntos", np.where(NPTS >= 30)[0]), ("limpias, ≥ 30 puntos", np.where((NPTS >= 30) & CLEAN)[0]),
                ("limpias reservadas, ≥ 30 puntos", np.intersect1d(CH, np.where(NPTS >= 30)[0]))]:
    d = (dl if "limpias" in nm else D_LOC)[idx]
    st = wilcoxon(d, alternative="greater") if len(d) > 5 and np.any(d != 0) else None
    rows.append(dict(grupo=nm, galaxias=len(idx), galaxias_mejor=f"{np.mean(d > 0):.0%}" if len(d) else "–",
                     p_Wilcoxon=f"{st.pvalue:.1e}" if st else "pocas"))
P3 = pd.DataFrame(rows); print(P3.to_string(index=False))
t_pts = REG.loc["log número de puntos", "t"]
print(f"\nEfecto del número de puntos controlando por polvo (regresión): t = {t_pts:+.1f}")

## 7. Veredicto (con las reglas fijadas al principio)

In [ ]:
def veredicto():
    if not HAVE_SPARC:
        print("⚠️  Datos SINTÉTICOS: solo comprueba que el código funciona.\n")
    dust_t = [REG.loc["inclinación", "t"], -REG.loc["tipo de Hubble T (bajo = más polvo)", "t"],
              REG.loc["log luminosidad 3.6 μm", "t"], -REG.loc["espaciado interior [″] (bajo = Hα)", "t"]]
    p1 = sum(t > 2.5 for t in dust_t) >= 2 or u.pvalue < 0.01
    pl = float(P2.p_Wilcoxon.iloc[0]) if P2.p_Wilcoxon.iloc[0] != "–" else 1.0
    p2_phys = (F_CL["lam"] > 0) and (pl < 0.05) and (P2.Δχ2_total.iloc[0] > 0)
    print(f"P1 indicadores de polvo: t (incl., tipo temprano, luminosidad, Hα) = {', '.join(f'{t:+.1f}' for t in dust_t)}; "
          f"riesgo > limpias p = {u.pvalue:.1e} → " + ("la mejora SIGUE al polvo" if p1 else "la mejora NO sigue al polvo"))
    print(f"P2 submuestra limpia: λ = {F_CL['lam']:.2f} kpc; reservada limpia p = {pl:.1e} → "
          + ("λ predice galaxias limpias" if p2_phys else "λ no predice galaxias limpias"))
    print(f"P3 número de puntos controlando por polvo: t = {t_pts:+.1f}")
    print()
    if p1 and not p2_phys:
        print("→ ARTEFACTO DEL POLVO (muy probable): la mejora de λ se concentra en galaxias con riesgo de polvo y desaparece en las limpias. "
              "La pista de los notebooks 4–6 queda explicada; la hipótesis de fondo no se ve afectada, pero esta señal deja de contar a su favor.")
    elif p2_phys:
        print("→ LA SEÑAL SOBREVIVE AL CONTROL DE POLVO: en galaxias limpias, λ se ajusta y predice el grupo reservado. "
              "Vale la pena registrar la hipótesis y probarla en una muestra externa de alta resolución.")
    elif not p1:
        print("→ EL POLVO NO LA EXPLICA, PERO TAMPOCO HAY SEÑAL LIMPIA: la mejora no sigue al polvo, y en las galaxias limpias λ tampoco predice. "
              "La señal parece ruido o un sistemático distinto (movimientos no circulares, distancias).")
    else:
        print("→ MIXTO: la mejora sigue en parte al polvo y la submuestra limpia no basta para decidir (pocas galaxias).")

veredicto()